# Complete SQL Tutorial

This notebook teaches SQL from beginner to advanced practical topics. It uses Python's built-in SQLite database so you can run the examples immediately, without installing MySQL, PostgreSQL, or SQL Server.

**What you will learn**

- Database and table basics
- `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`
- Aggregation with `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- `GROUP BY` and `HAVING`
- Joins: inner, left, self joins
- Subqueries and CTEs
- Window functions
- Table creation, inserts, updates, deletes
- Constraints, keys, indexes, transactions
- SQL for analytics and interview practice
- Best resources and project ideas

## 1. What Is SQL?

SQL means **Structured Query Language**. It is used to store, retrieve, filter, update, and analyze data in relational databases.

A relational database stores data in tables, similar to spreadsheets:

- A **table** stores one type of entity, such as customers or orders.
- A **row** is one record.
- A **column** is one attribute.
- A **primary key** uniquely identifies each row.
- A **foreign key** connects rows across tables.

### Popular SQL Databases

| Database | Common use |
|---|---|
| SQLite | Local apps, small projects, learning |
| PostgreSQL | Analytics, production apps, advanced SQL |
| MySQL | Web apps, business systems |
| SQL Server | Microsoft enterprise stack |
| Oracle | Large enterprise systems |
| DuckDB | Local analytics, data science |

Most basic SQL is similar across databases, but date functions, string functions, data types, and some advanced syntax can differ.

## 2. Setup: Run SQL Inside This Notebook

We will use `sqlite3`, which comes with Python. The helper function below runs SQL and displays results as a pandas DataFrame.

If you later use PostgreSQL or MySQL, the SQL concepts remain the same, but the connection tool changes.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")

def run_sql(query):
    """Run a SQL query and return results as a DataFrame when possible."""
    query = query.strip()
    try:
        return pd.read_sql_query(query, conn)
    except Exception:
        cursor = conn.executescript(query)
        conn.commit()
        return "Query executed successfully"

print("SQLite ready")

## 3. Create a Practice Database

We will use a small e-commerce style database:

- `customers`: people who buy products
- `products`: items for sale
- `orders`: each order placed by a customer
- `order_items`: products inside each order

This structure lets us practice real joins, aggregations, and analytics questions.

In [ ]:
run_sql("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL CHECK (price >= 0)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price REAL NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

In [ ]:
run_sql("""
INSERT INTO customers VALUES
(1, 'Aman', 'Delhi', '2025-01-05'),
(2, 'Sara', 'Mumbai', '2025-01-12'),
(3, 'Neha', 'Delhi', '2025-02-01'),
(4, 'Ravi', 'Pune', '2025-02-10'),
(5, 'Kiran', 'Mumbai', '2025-03-02');

INSERT INTO products VALUES
(1, 'Laptop', 'Electronics', 65000),
(2, 'Mouse', 'Electronics', 700),
(3, 'Keyboard', 'Electronics', 1500),
(4, 'Notebook', 'Stationery', 80),
(5, 'Pen', 'Stationery', 20),
(6, 'Office Chair', 'Furniture', 8500);

INSERT INTO orders VALUES
(101, 1, '2025-03-01', 'delivered'),
(102, 2, '2025-03-03', 'delivered'),
(103, 1, '2025-03-10', 'cancelled'),
(104, 3, '2025-03-12', 'delivered'),
(105, 4, '2025-03-15', 'shipped'),
(106, 2, '2025-03-18', 'delivered'),
(107, 5, '2025-03-20', 'pending');

INSERT INTO order_items VALUES
(1, 101, 1, 1, 65000),
(2, 101, 2, 2, 700),
(3, 102, 4, 10, 80),
(4, 102, 5, 20, 20),
(5, 103, 3, 1, 1500),
(6, 104, 6, 1, 8500),
(7, 105, 2, 3, 700),
(8, 106, 1, 1, 65000),
(9, 106, 3, 1, 1500),
(10, 107, 5, 5, 20);
""")

## 4. SELECT: Read Data From a Table

`SELECT` chooses columns from a table.

```sql
SELECT column1, column2
FROM table_name;
```

Use `*` to select all columns, but in production code it is better to name the columns you need.

In [ ]:
run_sql("""
SELECT customer_id, name, city
FROM customers;
""")

## 5. WHERE: Filter Rows

`WHERE` keeps only rows that match a condition.

Common operators:

| Operator | Meaning |
|---|---|
| `=` | equals |
| `<>` or `!=` | not equal |
| `>` `<` `>=` `<=` | comparisons |
| `BETWEEN` | inside range |
| `IN` | matches any value in a list |
| `LIKE` | pattern matching |
| `IS NULL` | missing value check |

In [ ]:
run_sql("""
SELECT product_name, category, price
FROM products
WHERE category = 'Electronics' AND price > 1000;
""")

In [ ]:
run_sql("""
SELECT name, city
FROM customers
WHERE city IN ('Delhi', 'Pune');
""")

## 6. ORDER BY and LIMIT

`ORDER BY` sorts results. `LIMIT` restricts how many rows are returned.

Default sort is ascending. Use `DESC` for descending.

In [ ]:
run_sql("""
SELECT product_name, price
FROM products
ORDER BY price DESC
LIMIT 3;
""")

## 7. Calculated Columns and Aliases

You can create calculated columns in a query. `AS` gives a readable column name.

In [ ]:
run_sql("""
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    quantity * unit_price AS line_total
FROM order_items;
""")

## 8. Aggregate Functions

Aggregates summarize many rows into one value.

Common functions:

- `COUNT(*)`: number of rows
- `SUM(column)`: total
- `AVG(column)`: average
- `MIN(column)`: smallest value
- `MAX(column)`: largest value

In [ ]:
run_sql("""
SELECT
    COUNT(*) AS total_products,
    AVG(price) AS average_price,
    MIN(price) AS cheapest,
    MAX(price) AS most_expensive
FROM products;
""")

## 9. GROUP BY

`GROUP BY` creates one summary row per group.

Rule: every selected column must either be grouped or aggregated.

In [ ]:
run_sql("""
SELECT
    category,
    COUNT(*) AS product_count,
    ROUND(AVG(price), 2) AS avg_price
FROM products
GROUP BY category
ORDER BY avg_price DESC;
""")

## 10. HAVING

`WHERE` filters rows before grouping. `HAVING` filters groups after aggregation.

In [ ]:
run_sql("""
SELECT
    category,
    COUNT(*) AS product_count,
    AVG(price) AS avg_price
FROM products
GROUP BY category
HAVING COUNT(*) >= 2;
""")

## 11. SQL Logical Order

SQL is written in one order but logically processed in another.

Written order:

```sql
SELECT
FROM
WHERE
GROUP BY
HAVING
ORDER BY
LIMIT
```

Logical order:

```text
FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY -> LIMIT
```

This explains why you usually cannot use a `SELECT` alias inside `WHERE`.

## 12. INNER JOIN

A join combines rows from multiple tables.

`INNER JOIN` returns only matching rows from both tables.

Example: show each order with the customer name.

In [ ]:
run_sql("""
SELECT
    o.order_id,
    c.name,
    c.city,
    o.order_date,
    o.status
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")

## 13. Multiple Joins

Real analytics often needs 3 or more tables.

Question: What is the revenue per order?

In [ ]:
run_sql("""
SELECT
    o.order_id,
    c.name,
    o.status,
    SUM(oi.quantity * oi.unit_price) AS order_revenue
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY o.order_id, c.name, o.status
ORDER BY order_revenue DESC;
""")

## 14. LEFT JOIN

`LEFT JOIN` returns all rows from the left table, plus matching rows from the right table.

Use it when you want to keep unmatched rows.

Example: show all customers, including customers who have not ordered.

In [ ]:
run_sql("""
SELECT
    c.customer_id,
    c.name,
    COUNT(o.order_id) AS order_count
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id;
""")

## 15. NULL Values

`NULL` means unknown or missing. It is not equal to zero or an empty string.

Use:

```sql
IS NULL
IS NOT NULL
```

Do not use:

```sql
= NULL
```

In [ ]:
run_sql("""
SELECT customer_id, name, city
FROM customers
WHERE city IS NOT NULL;
""")

## 16. CASE WHEN

`CASE` creates conditional logic inside SQL. It is like `if/elif/else` in Python.

In [ ]:
run_sql("""
SELECT
    product_name,
    price,
    CASE
        WHEN price >= 10000 THEN 'high'
        WHEN price >= 1000 THEN 'medium'
        ELSE 'low'
    END AS price_band
FROM products
ORDER BY price DESC;
""")

## 17. Subqueries

A subquery is a query inside another query.

Example: find products whose price is above the average product price.

In [ ]:
run_sql("""
SELECT product_name, price
FROM products
WHERE price > (
    SELECT AVG(price)
    FROM products
)
ORDER BY price DESC;
""")

## 18. CTEs: Common Table Expressions

A CTE creates a named temporary result using `WITH`. CTEs make complex SQL easier to read.

Question: What are the delivered-order revenues by customer?

In [ ]:
run_sql("""
WITH order_revenue AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders AS o
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id
)
SELECT
    c.name,
    SUM(orv.revenue) AS total_delivered_revenue
FROM order_revenue AS orv
JOIN customers AS c
    ON orv.customer_id = c.customer_id
GROUP BY c.name
ORDER BY total_delivered_revenue DESC;
""")

## 19. Window Functions

Window functions calculate values across related rows without collapsing rows like `GROUP BY` does.

Common window functions:

- `ROW_NUMBER()`
- `RANK()`
- `DENSE_RANK()`
- `SUM() OVER (...)`
- `AVG() OVER (...)`
- `LAG()` and `LEAD()`

Question: rank products by price inside each category.

In [ ]:
run_sql("""
SELECT
    category,
    product_name,
    price,
    RANK() OVER (PARTITION BY category ORDER BY price DESC) AS price_rank
FROM products
ORDER BY category, price_rank;
""")

In [ ]:
run_sql("""
WITH daily_revenue AS (
    SELECT
        o.order_date,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders AS o
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.order_date
)
SELECT
    order_date,
    revenue,
    SUM(revenue) OVER (ORDER BY order_date) AS running_revenue
FROM daily_revenue;
""")

## 20. INSERT, UPDATE, DELETE

These commands change data.

- `INSERT`: add rows
- `UPDATE`: modify rows
- `DELETE`: remove rows

Always use a careful `WHERE` clause for `UPDATE` and `DELETE`. Without `WHERE`, you may update or delete every row.

In [ ]:
run_sql("""
INSERT INTO products (product_id, product_name, category, price)
VALUES (7, 'Desk Lamp', 'Furniture', 1200);

UPDATE products
SET price = 1100
WHERE product_id = 7;

SELECT *
FROM products
WHERE product_id = 7;
""")

## 21. Constraints and Keys

Constraints protect data quality.

Common constraints:

| Constraint | Purpose |
|---|---|
| `PRIMARY KEY` | unique row identifier |
| `FOREIGN KEY` | links to another table |
| `NOT NULL` | value is required |
| `UNIQUE` | no duplicates allowed |
| `CHECK` | rule for allowed values |
| `DEFAULT` | automatic value if none is given |

A good database design prevents bad data before it enters the system.

## 22. Indexes

An index helps the database find rows faster, like an index at the back of a book.

Indexes are useful for columns used often in:

- `WHERE`
- `JOIN`
- `ORDER BY`
- `GROUP BY`

Tradeoff: indexes speed up reads but slow down inserts/updates and use extra storage.

In [ ]:
run_sql("""
CREATE INDEX IF NOT EXISTS idx_orders_customer_id
ON orders(customer_id);

CREATE INDEX IF NOT EXISTS idx_orders_order_date
ON orders(order_date);
""")

## 23. Transactions

A transaction groups multiple changes into one unit.

Core idea: either all changes succeed, or none are saved.

Commands:

```sql
BEGIN;
UPDATE ...;
INSERT ...;
COMMIT;
```

If something goes wrong:

```sql
ROLLBACK;
```

Transactions protect money transfers, inventory updates, order creation, and other important workflows.

## 24. Database Normalization

Normalization means organizing tables to reduce duplication and improve consistency.

### Common Normal Forms

- **1NF**: each cell contains one value, no repeated groups.
- **2NF**: every non-key column depends on the whole primary key.
- **3NF**: non-key columns depend only on the key, not on other non-key columns.

For analytics, teams sometimes use denormalized tables for speed and convenience. For transactional systems, normalized design is usually safer.

## 25. SQL for Data Analysis

Useful analytics questions:

- Total revenue by month
- Top customers by revenue
- Conversion rate by campaign
- Average order value
- Repeat purchase rate
- Cohort retention
- Most common product combinations
- Churned customers

SQL is often the first tool used by data analysts, data scientists, and data engineers before Python or BI dashboards.

In [ ]:
run_sql("""
SELECT
    c.name,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(oi.quantity * oi.unit_price) AS gross_revenue,
    ROUND(AVG(oi.quantity * oi.unit_price), 2) AS avg_line_value
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.status <> 'cancelled'
GROUP BY c.name
ORDER BY gross_revenue DESC;
""")

## 26. Common SQL Mistakes

Watch out for these:

- Forgetting `WHERE` in `UPDATE` or `DELETE`
- Using `= NULL` instead of `IS NULL`
- Joining on the wrong key
- Creating duplicate rows by joining many-to-many tables carelessly
- Filtering after a `LEFT JOIN` in a way that accidentally turns it into an inner join
- Selecting non-grouped columns with aggregates
- Confusing `WHERE` and `HAVING`
- Assuming SQL dialects all use the same date functions
- Using `SELECT *` in production queries
- Not checking row counts after joins

## 27. Practice Questions

Try writing SQL for these tasks:

1. Show all products cheaper than 1000.
2. Show customers from Mumbai.
3. Count orders by status.
4. Calculate revenue by product category.
5. Find the top 3 products by total quantity sold.
6. Find customers with more than one order.
7. Show orders with customer name and total order amount.
8. Find products never ordered.
9. Rank customers by total revenue.
10. Calculate running revenue by date.
11. Find each customer's first order date.
12. Find average order value excluding cancelled orders.
13. Find the most expensive product in each category.
14. Find categories with total revenue above 5000.
15. Create a new table for suppliers and connect products to suppliers.

## 28. Mini Projects

### Beginner Projects

- Student marks database
- Library management database
- Sales analysis dashboard queries
- Employee department database
- Movie database queries

### Intermediate Projects

- E-commerce analytics project
- Hospital appointment database
- Bank transaction analysis
- Customer churn SQL analysis
- Inventory and order management system

### Advanced Projects

- Cohort retention analysis
- Fraud transaction investigation
- SQL interview question bank
- Data warehouse star schema
- ETL pipeline using SQL plus Python
- Performance tuning with indexes and query plans

## 29. Best Resources

Use these resources in this order:

1. **SQLBolt**: short interactive SQL lessons with exercises.  
   https://sqlbolt.com/

2. **Mode SQL Tutorial**: excellent SQL for data analysis.  
   https://mode.com/sql-tutorial/

3. **PostgreSQL Official Tutorial**: official relational database and SQL tutorial.  
   https://www.postgresql.org/docs/current/tutorial.html

4. **SQLite Documentation**: useful because this notebook uses SQLite.  
   https://sqlite.org/docs.html

5. **W3Schools SQL Tutorial**: quick syntax reference and simple examples.  
   https://www.w3schools.com/sql/

6. **DataLemur SQL Tutorial**: practical SQL for data science and interviews.  
   https://datalemur.com/sql-tutorial

7. **LeetCode SQL 50**: SQL interview practice.  
   https://leetcode.com/studyplan/top-sql-50/

8. **HackerRank SQL**: beginner-to-intermediate SQL practice.  
   https://www.hackerrank.com/domains/sql

### Suggested Learning Order

1. SELECT, WHERE, ORDER BY, LIMIT
2. Aggregates, GROUP BY, HAVING
3. Joins
4. Subqueries and CTEs
5. Window functions
6. Table design and constraints
7. Indexes and query performance
8. Transactions
9. Real analytics projects
10. Interview practice

## 30. SQL Interview Checklist

You should be able to explain and use:

- What is SQL?
- What is a relational database?
- Primary key vs foreign key
- `WHERE` vs `HAVING`
- `INNER JOIN` vs `LEFT JOIN`
- `COUNT(*)` vs `COUNT(column)`
- `UNION` vs `UNION ALL`
- `DELETE` vs `TRUNCATE` vs `DROP`
- Normalization and denormalization
- Indexes and their tradeoffs
- Transactions and ACID
- CTEs vs subqueries
- Window functions
- Duplicate handling
- NULL handling
- Query debugging

Final advice: SQL skill grows fastest when you ask real business questions and force yourself to answer them from tables.